# Evolution Dataset — Generation Pipeline V5

This notebook implements a synthetic dataset construction pipeline for the ADL evolution task. Given an ADL snippet (ACME or AADL) and a natural-language evolution request, the model must produce the modified ADL code.

**Taxonomy — 6 operations across 3 architectural levels:**

| Level | Operation | Difficulty |
|---|---|---|
| Interface | ADD_PORT | Easy |
| Interface | MODIFY_PROPERTY | Easy |
| Component | ADD_COMPONENT | Medium |
| Component | DELETE_COMPONENT | Hard |
| Connector | ADD_CONNECTOR | Medium |
| Connector | DELETE_CONNECTOR | Hard |

**Input format:** `evolve: [OP] {operation} [ADL] {adl_code} [REQ] {natural_language_request}`

> **Note:** Results are exported to `EVO_DIR` as `.csv`, `.jsonl`, and a structured `rapport_evolution_v5.json` report. Run cells in order.

## 1. Setup

In [ ]:
# Mount Google Drive and configure I/O paths.
from google.colab import drive
import os

drive.mount('/content/drive')

BASE_DIR   = "/content/drive/MyDrive"
CLEAN_DIR  = f"{BASE_DIR}/dataset_clean"
EVO_DIR    = f"{CLEAN_DIR}/evolution"
INPUT_PATH = f"{CLEAN_DIR}/dataset_Comprehension.csv"

os.makedirs(EVO_DIR, exist_ok=True)

if os.path.exists(INPUT_PATH):
    print(f"Source dataset located : {INPUT_PATH}")
else:
    raise FileNotFoundError(
        f"Dataset not found at {INPUT_PATH}. "
        "Please verify that dataset_Comprehension.csv exists under CLEAN_DIR."
    )
print(f"Output directory       : {EVO_DIR}")

## 2. Imports and Dataset Loading

In [ ]:
import pandas as pd
import json
import re
import random
from collections import defaultdict
from sklearn.model_selection import train_test_split

random.seed(42)

df = pd.read_csv(INPUT_PATH, encoding="utf-8")

# Normalise column name for quality score
if "quality_score" in df.columns and "quality" not in df.columns:
    df = df.rename(columns={"quality_score": "quality"})

print(f"Source dataset loaded: {len(df)} samples")
print(f"  ACME : {len(df[df['adl_type']=='ACME'])}")
print(f"  AADL : {len(df[df['adl_type']=='AADL'])}")
print(f"  Columns : {list(df.columns)}")

## 3. Architectural Catalogues

The following catalogues define the vocabulary used across all six generators:
property names and values per element type (for MODIFY_PROPERTY), component and connector type pools per architectural pattern (for ADD_* operations), port name lists (for ADD_PORT), and natural-language paraphrase templates (for all operations).

In [ ]:
# ===========================================================================
# Property catalogues — used by MODIFY_PROPERTY generators
# ===========================================================================
PROPERTIES_ACME = {
    "Component": [
        ("reliability",    ["0.99", "0.95", "0.999", "0.9999"]),
        ("throughput",     ["100msg/s", "500msg/s", "1000msg/s"]),
        ("latency",        ["10ms", "50ms", "100ms", "5ms"]),
        ("availability",   ["0.99", "0.999", "0.9999"]),
        ("maxConnections", ["10", "50", "100", "500"]),
        ("priority",       ["1", "5", "10", "15"]),
    ],
    "Connector": [
        ("protocol",   ["TCP", "UDP", "HTTP", "RPC", "AMQP"]),
        ("bandwidth",  ["1Gbps", "100Mbps", "10Gbps"]),
        ("synchrony",  ["synchronous", "asynchronous"]),
        ("buffered",   ["true", "false"]),
        ("maxLatency", ["5ms", "10ms", "50ms", "100ms"]),
    ],
    "System": [
        ("version",       ["1.0", "2.0", "1.5", "3.0"]),
        ("deploymentEnv", ["cloud", "embedded", "desktop", "server"]),
    ]
}

PROPERTIES_AADL = {
    "thread": [
        ("Dispatch_Protocol", ["Periodic", "Sporadic", "Aperiodic"]),
        ("Period",            ["10ms", "20ms", "50ms", "100ms"]),
        ("Priority",          ["1", "5", "10", "15", "20"]),
        ("Stack_Size",        ["4096 bytes", "8192 bytes", "16384 bytes"]),
    ],
    "process": [
        ("Runtime_Protection",  ["true", "false"]),
        ("Scheduling_Protocol", ["FIFO_Within_Priorities", "Round_Robin_Protocol"]),
    ],
    "system": [
        ("Security_Level",  ["low", "medium", "high", "critical"]),
        ("Fault_Tolerance", ["true", "false"]),
    ],
    "device": [
        ("SEI::MIPSCapacity", ["100.0 MIPS", "200.0 MIPS", "500.0 MIPS"]),
    ]
}


# ===========================================================================
# Component and connector type pools per architectural pattern
# ===========================================================================
TYPES_BY_PATTERN = {
    "client_server": {
        "Component": ["Cache", "LoadBalancer", "Proxy", "Gateway", "Authenticator", "SessionManager"],
        "Connector": ["RPCConnector", "HTTPConnector", "MessageQueue", "ServiceLink"]
    },
    "pipe_filter": {
        "Component": ["Splitter", "Merger", "Validator", "Transformer", "Aggregator"],
        "Connector": ["BufferedPipe", "SyncPipe", "AsyncPipe", "StreamPipe"]
    },
    "event_driven": {
        "Component": ["EventHandler", "Subscriber", "Publisher", "Dispatcher"],
        "Connector": ["EventChannel", "TopicBus", "NotificationBus", "SignalWire"]
    },
    "layered": {
        "Component": ["ServiceLayer", "DataLayer", "PresentationLayer", "APILayer"],
        "Connector": ["LayerBridge", "APIGateway", "DataBus", "ServiceBridge"]
    },
    "generic": {
        "Component": ["Logger", "Monitor", "Auditor", "Validator", "Authenticator",
                       "HealthChecker", "ConfigManager"],
        "Connector": ["Pipe", "Bus", "Channel", "Link", "Bridge", "Wire"]
    }
}

# Connector pools — (TypeName, instanceName) pairs for ACME;
#                   (TypeName, instanceName, aadlCategory) triples for AADL
CONNECTORS_ACME = [
    ("Pipe",          "pipe1"),
    ("Bus",           "bus1"),
    ("EventChannel",  "eventChannel1"),
    ("RPCConnector",  "rpcConn1"),
    ("AsyncPipe",     "asyncPipe1"),
    ("MessageQueue",  "msgQueue1"),
    ("DataBridge",    "dataBridge1"),
    ("ServiceLink",   "serviceLink1"),
    ("HTTPConnector", "httpConn1"),
    ("StreamPipe",    "streamPipe1"),
]

CONNECTORS_AADL = [
    ("DataBus",     "dataBus1",     "bus"),
    ("Channel",     "channel1",     "bus"),
    ("Bridge",      "bridge1",      "bus"),
    ("EventBus",    "eventBus1",    "bus"),
    ("ControlLink", "ctrlLink1",    "bus"),
    ("ServiceBus",  "serviceBus1",  "bus"),
    ("NetworkLink", "netLink1",     "bus"),
    ("SharedBus",   "sharedBus1",   "bus"),
]

# Port name pools
PORTS_ACME = [
    "monitoringPort", "statusPort", "controlPort", "dataPort",
    "errorPort", "syncPort", "healthPort", "auditPort",
    "notifyPort", "diagnosticPort", "logPort", "heartbeatPort",
    "feedbackPort", "triggerPort", "alertPort",
]
PORTS_AADL = [
    "status_port", "control_port", "data_port", "error_port",
    "sync_port", "health_port", "audit_port", "notify_port",
    "monitor_port", "diagnostic_port", "log_port", "heartbeat_port",
    "feedback_port", "trigger_port", "alert_port",
]

# Natural-language paraphrase templates per operation
PARAPHRASES = {
    "ADD_PORT": [
        "Add a {port} port to the {nom} component",
        "Insert a new {port} interface into {nom}",
        "Extend {nom} with a {port} port",
        "Create a {port} port for the {nom} component",
        "Introduce a {port} capability to {nom}",
        "Attach a {port} port to the {nom} component",
        "Augment {nom} with a new {port} interface",
    ],
    "MODIFY_PROPERTY": [
        "Change the {prop} of {nom} to {val}",
        "Update the {prop} property of {nom} to {val}",
        "Set {prop} of {nom} to {val}",
        "Modify the {prop} attribute of {nom} to {val}",
        "Update {nom} {prop} value to {val}",
        "Change {nom}'s {prop} to {val}",
        "Set the {prop} property of {nom} to the value {val}",
    ],
    "ADD_COMPONENT": [
        "Add a {type} component named {nom} to the {system} system",
        "Insert a new {nom} component of type {type} into {system}",
        "Extend the {system} architecture with a {type} component named {nom}",
        "Introduce a {nom} {type} component to the {system} system",
        "Add a new {nom} of type {type} to {system}",
        "Create a {type} component called {nom} in {system}",
        "Append a {type} component {nom} to the {system} architecture",
    ],
    "DELETE_COMPONENT": [
        "Delete the {nom} component from the architecture",
        "Remove component {nom} from the system",
        "Eliminate the {nom} component from the design",
        "Drop component {nom} from the ADL specification",
        "Remove the {nom} component from the architecture",
        "Delete {nom} from the architectural specification",
        "Strip the {nom} component out of the architecture",
    ],
    "ADD_CONNECTOR": [
        "Add a {type} connector named {nom} to the {system} system",
        "Insert a new {nom} connector of type {type} into {system}",
        "Extend the {system} architecture with a {type} connector named {nom}",
        "Create a {type} connector called {nom} in the {system} system",
        "Add a new connector {nom} of type {type} to {system}",
        "Append a {type} connector {nom} to the {system} architecture",
        "Introduce a {nom} {type} connector to the {system} system",
    ],
    "DELETE_CONNECTOR": [
        "Delete the {nom} connector from the architecture",
        "Remove connector {nom} from the system",
        "Eliminate the {nom} connector from the design",
        "Drop the {nom} connector from the ADL specification",
        "Remove the {nom} connector from the architecture",
        "Delete connector {nom} from the architectural specification",
        "Strip the {nom} connector out of the architecture",
    ],
}

print(f"Catalogues loaded:")
print(f"  ACME connector types : {len(CONNECTORS_ACME)}")
print(f"  AADL connector types : {len(CONNECTORS_AADL)}")
print(f"  ACME / AADL ports    : {len(PORTS_ACME)} / {len(PORTS_AADL)}")
print(f"  Paraphrase templates : {sum(len(v) for v in PARAPHRASES.values())} total")

## 4. Utility Functions

Helper functions for name extraction, architectural-pattern detection, input formatting, system-block filtering, validation, and property injection.

In [ ]:
# ===========================================================================
# Stopword list — prevents generic keywords from being treated as entity names
# ===========================================================================
STOPWORDS = {
    "type", "new", "extended", "with", "implementation",
    "base", "in", "out", "i", "o", "p", "r", "a", "an",
    "the", "end", "is", "of", "to", "for", "and", "or",
    "not", "if", "else", "then", "do", "while", "none"
}


def extract_name(code: str) -> str:
    """Extract the primary entity name from an ADL snippet."""
    patterns = [
        r'(?:Component|Connector|System|Style|Family)(?:\s+Type)?\s+(\w+)',
        r'(?:Component|Connector|System)\s+(\w+)\s*[=:{]',
        r'(?:process|thread|system|device|memory|bus)\s+implementation\s+(\w+)',
        r'(?:process|thread|system|device|memory|bus)\s+(\w+)\b',
        r'(\w+)\s*:\s*(?:process|thread|device|memory)\s+\w+\s*;',
    ]
    for p in patterns:
        m = re.search(p, code, re.IGNORECASE)
        if m:
            name = m.group(1)
            if name.lower() not in STOPWORDS and len(name) > 2:
                return name
    return None


def extract_system_name(code: str) -> str:
    """Extract the top-level system name from an ADL snippet."""
    for p in [
        r'System\s+(\w+)\s*[=:{]',
        r'system\s+implementation\s+(\w+)',
        r'bus\s+implementation\s+(\w+)',
        r'process\s+implementation\s+(\w+)',
        r'system\s+(\w+)\b',
    ]:
        m = re.search(p, code, re.IGNORECASE)
        if m:
            name = m.group(1)
            if name.lower() not in STOPWORDS:
                return name
    return "MainSystem"


def detect_pattern(code: str) -> str:
    """Classify an ADL snippet into one of five architectural patterns."""
    c = code.lower()
    if any(w in c for w in ["client", "server", "request", "response", "rpc"]):
        return "client_server"
    if any(w in c for w in ["filter", "pipe", "source", "sink", "stream"]):
        return "pipe_filter"
    if any(w in c for w in ["event", "handler", "subscriber", "publisher"]):
        return "event_driven"
    if any(w in c for w in ["layer", "presentation", "service", "business"]):
        return "layered"
    return "generic"


def fmt_input(operation: str, adl_code: str, request: str) -> str:
    """Format an input sample in the standard [OP][ADL][REQ] template."""
    return f"evolve: [OP] {operation} [ADL] {adl_code} [REQ] {request}"


def is_system_block(code: str) -> bool:
    """
    Return True if the snippet represents a complete architectural system
    (System declaration or implementation block) where connectors can be
    meaningfully inserted. Isolated Port and Component Type declarations
    are excluded.
    """
    c = code.lower().strip()
    system_patterns = [
        r'\bsystem\s+\w+',
        r'\bsystem\s+implementation\b',
        r'\bprocess\s+implementation\b',
        r'\bbus\s+implementation\b',
        r'\bdevice\s+implementation\b',
        r'\bsubcomponents\b',
        r'\battachments\s*\{',
    ]
    for p in system_patterns:
        if re.search(p, c):
            return True
    # ACME systems with at least two declared components
    return len(re.findall(r'\bcomponent\b', c, re.IGNORECASE)) >= 2


# ===========================================================================
# Validation functions
# ===========================================================================
def validate(sample: dict) -> bool:
    """Apply five generic validation rules to a generated sample."""
    target    = str(sample.get("target", "")).strip()
    adl_avant = str(sample.get("adl_avant", "")).strip()
    operation = sample.get("operation", "")

    # R1 — non-trivial target
    if len(target.split()) < 3:
        return False
    # R2 — target differs from source
    if target == adl_avant:
        return False
    # R3 — balanced braces for ACME targets
    if sample.get("adl_type") == "ACME":
        diff = target.count("{") - target.count("}")
        if diff != 0:
            if 0 < diff <= 2:
                sample["target"] = target + "}" * diff
            else:
                return False
    # R4 — deleted entity absent from target
    if operation in ["DELETE_COMPONENT", "DELETE_CONNECTOR"]:
        cible = sample.get("cible", "")
        if cible and re.search(rf'\b{re.escape(cible)}\b', target):
            return False
    # R5 — standard input format
    if not all(t in sample.get("input", "") for t in ["[OP]", "[ADL]", "[REQ]"]):
        return False
    return True


def validate_add_connector(sample: dict) -> bool:
    """
    Apply five strict validation rules specific to ADD_CONNECTOR samples.
    Ensures that the target contains a real, newly inserted connector.
    """
    target    = str(sample.get("target", "")).strip()
    adl_avant = str(sample.get("adl_avant", "")).strip()
    adl_type  = sample.get("adl_type", "")
    conn_name = sample.get("cible_conn", "")

    # R1 — non-trivial target (at least 10 tokens)
    if len(target.split()) < 10:
        return False
    # R2 — target differs from source
    if target.strip() == adl_avant.strip():
        return False
    # R3 — connector name present in target
    if conn_name and conn_name.lower() not in target.lower():
        return False
    # R4 — connector keyword present
    target_l = target.lower()
    if adl_type == "ACME":
        if "connector" not in target_l:
            return False
        diff = target.count("{") - target.count("}")
        if diff != 0:
            if 0 < diff <= 2:
                sample["target"] = target + "}" * diff
            else:
                return False
    else:
        if not any(kw in target_l for kw in ["connections", "bus access", "connector"]):
            return False
    # R5 — target longer than source (something was added)
    if len(target) <= len(adl_avant):
        return False
    return True


# ===========================================================================
# Property injection helpers
# Used when a snippet contains no existing property to modify.
# ===========================================================================
def inject_property_acme(code: str, element_type: str):
    """Inject a synthetic property block into an ACME snippet."""
    props = PROPERTIES_ACME.get(element_type, PROPERTIES_ACME["Component"])
    prop_name, values = random.choice(props)
    val_before = values[0]
    others     = [v for v in values if v != val_before]
    val_after  = random.choice(others) if others else values[-1]
    block      = f"\n  Properties {{\n    {prop_name} = {val_before};\n  }}"
    injected   = re.sub(r'(\})\s*$', block + r'\n}', code, count=1)
    return injected, prop_name, val_before, val_after


def inject_property_aadl(code: str):
    """Inject a synthetic property block into an AADL snippet."""
    aadl_type = "thread"
    for t in ["thread", "process", "system", "device"]:
        if re.search(rf'\b{t}\b', code, re.IGNORECASE):
            aadl_type = t
            break
    props      = PROPERTIES_AADL.get(aadl_type, PROPERTIES_AADL["thread"])
    prop_name, values = random.choice(props)
    val_before = values[0]
    others     = [v for v in values if v != val_before]
    val_after  = random.choice(others) if others else values[-1]
    block      = f"\n  properties\n    {prop_name} => {val_before};"
    injected   = re.sub(r'(\bend\b)', block + r'\nend', code, count=1)
    if injected == code:
        injected = code + block
    return injected, prop_name, val_before, val_after


print("Utility functions defined.")

## 5. Sample Generators (Operations 1–6)

Each generator takes a block dictionary `{adl_code, adl_type, element_type, quality}` and returns a list of validated `{input, target, operation, adl_type, difficulte, source}` dictionaries.

**ADD_CONNECTOR** applies only to complete system blocks (`is_system_block` filter) and is subject to five additional validation rules.

In [ ]:
# ===========================================================================
# Operation 1 — ADD_PORT
# ===========================================================================
def gen_add_port(block: dict) -> list:
    """Generate ADD_PORT samples by inserting a new port into a component snippet."""
    code     = block["adl_code"]
    adl_type = block["adl_type"]
    name     = extract_name(code)
    if not name:
        return []
    results = []
    ports   = PORTS_ACME if adl_type == "ACME" else PORTS_AADL
    for new_port in random.sample(ports, min(3, len(ports))):
        if adl_type == "ACME":
            port_code = f"\n  Port {new_port};"
            modified  = re.sub(r'(\})\s*$', port_code + r'\n}', code, count=1)
        else:
            port_code = f"\n  port {new_port} : requires data access;"
            modified  = re.sub(r'(\bend\b)', port_code + r'\nend', code, count=1)
            if modified == code:
                modified = code + port_code
        if modified == code:
            continue
        for tmpl in random.sample(PARAPHRASES["ADD_PORT"], min(2, len(PARAPHRASES["ADD_PORT"]))):
            request = tmpl.format(port=new_port, nom=name)
            s = {"adl_avant": code,
                 "input"    : fmt_input("ADD_PORT", code, request),
                 "target"   : modified,
                 "operation": "ADD_PORT",
                 "adl_type" : adl_type,
                 "difficulte": "easy",
                 "cible"    : name,
                 "source"   : "comp800_v5"}
            if validate(s):
                results.append(s)
    return results


# ===========================================================================
# Operation 2 — MODIFY_PROPERTY
# ===========================================================================
def gen_modify_property(block: dict) -> list:
    """Generate MODIFY_PROPERTY samples by updating existing or injected properties."""
    code     = block["adl_code"]
    adl_type = block["adl_type"]
    name     = extract_name(code)
    if not name:
        return []
    results = []

    def _make_sample(adl_src, prop, v_before, v_after, src_tag):
        modified = re.sub(
            rf'{re.escape(prop)}\s*[=>]+\s*{re.escape(v_before)}',
            f"{prop} {'=' if adl_type == 'ACME' else '=>'} {v_after}",
            adl_src, count=1)
        if modified == adl_src:
            return []
        out = []
        for tmpl in random.sample(PARAPHRASES["MODIFY_PROPERTY"], min(2, 7)):
            request = tmpl.format(prop=prop, nom=name, val=v_after)
            s = {"adl_avant" : adl_src,
                 "input"     : fmt_input("MODIFY_PROPERTY", adl_src, request),
                 "target"    : modified,
                 "operation" : "MODIFY_PROPERTY",
                 "adl_type"  : adl_type,
                 "difficulte": "easy",
                 "cible"     : name,
                 "source"    : src_tag}
            if validate(s):
                out.append(s)
        return out

    if adl_type == "ACME":
        existing = [
            (p, v) for p, v in re.findall(r'(\w+)\s*=\s*([\w".]+)\s*;', code)
            if p.lower() not in STOPWORDS and len(p) > 2
        ]
        if existing:
            for prop, val_before in existing[:2]:
                alternatives = next(
                    ([v for v in vals if v != val_before]
                     for _, props in PROPERTIES_ACME.items()
                     for p, vals in props if p.lower() == prop.lower()),
                    ["updated_value", "0.95", "true"]
                )
                results.extend(_make_sample(
                    code, prop, val_before,
                    random.choice(alternatives), "comp800_v5"))
        else:
            inj, prop, v_b, v_a = inject_property_acme(
                code, block.get("element_type", "Component"))
            results.extend(_make_sample(inj, prop, v_b, v_a, "comp800_v5_injected"))
    else:
        existing = [
            (p, v) for p, v in re.findall(r'([\w:]+)\s*=>\s*([\w".()]+)\s*;', code)
            if p.lower() not in STOPWORDS and len(p) > 2
        ]
        if existing:
            for prop, val_before in existing[:2]:
                alternatives = next(
                    ([v for v in vals if v != val_before]
                     for _, props in PROPERTIES_AADL.items()
                     for p, vals in props if p.lower() == prop.lower()),
                    ["Sporadic", "100ms", "high"]
                )
                results.extend(_make_sample(
                    code, prop, val_before,
                    random.choice(alternatives), "comp800_v5"))
        else:
            inj, prop, v_b, v_a = inject_property_aadl(code)
            results.extend(_make_sample(inj, prop, v_b, v_a, "comp800_v5_injected"))
    return results


# ===========================================================================
# Operation 3 — ADD_COMPONENT
# ===========================================================================
def gen_add_component(block: dict) -> list:
    """Generate ADD_COMPONENT samples by inserting a new component into a system snippet."""
    code     = block["adl_code"]
    adl_type = block["adl_type"]
    system   = extract_system_name(code)
    pattern  = detect_pattern(code)
    comp_types = TYPES_BY_PATTERN.get(pattern, TYPES_BY_PATTERN["generic"])["Component"]
    results  = []
    for comp_type in random.sample(comp_types, min(2, len(comp_types))):
        comp_name = comp_type[0].lower() + comp_type[1:] + "1"
        if adl_type == "ACME":
            decl = f"\n  Component {comp_name} : {comp_type};"
            if "Attachments" in code:
                modified = re.sub(r'(Attachments\s*\{)', decl + r'\n  \1', code, count=1)
            else:
                modified = re.sub(r'(\})\s*$', decl + r'\n}', code, count=1)
        else:
            decl = f"\n  subcomponents\n    {comp_name.lower()} : process {comp_type};"
            if "subcomponents" in code:
                modified = re.sub(
                    r'(subcomponents\b[^;]*;)',
                    r'\1' + f"\n    {comp_name.lower()} : process {comp_type};",
                    code, count=1, flags=re.DOTALL)
            else:
                modified = re.sub(r'(\bend\b)', decl + r'\nend', code, count=1)
                if modified == code:
                    modified = code + decl
        if modified == code:
            continue
        for tmpl in random.sample(PARAPHRASES["ADD_COMPONENT"], min(2, 7)):
            request = tmpl.format(type=comp_type, nom=comp_name, system=system)
            s = {"adl_avant" : code,
                 "input"     : fmt_input("ADD_COMPONENT", code, request),
                 "target"    : modified,
                 "operation" : "ADD_COMPONENT",
                 "adl_type"  : adl_type,
                 "difficulte": "medium",
                 "cible"     : system,
                 "source"    : "comp800_v5"}
            if validate(s):
                results.append(s)
    return results


# ===========================================================================
# Operation 4 — DELETE_COMPONENT
# ===========================================================================
def gen_delete_component(block: dict) -> list:
    """Generate DELETE_COMPONENT samples by removing an identified component."""
    code     = block["adl_code"]
    adl_type = block["adl_type"]
    system   = extract_system_name(code)
    if adl_type == "ACME":
        candidates = re.findall(r'Component(?:\s+Type)?\s+(\w+)', code, re.IGNORECASE)
    else:
        candidates = re.findall(
            r'(\w+)\s*:\s*(?:process|thread|device|memory|system|bus)\s+\w+', code)
        if not candidates:
            candidates = re.findall(r'(?:process|thread|device|system)\s+(\w+)\b', code)
    candidates = [c for c in candidates if c.lower() not in STOPWORDS and len(c) > 2]
    if not candidates:
        return []
    results = []
    for target_name in candidates[:3]:
        modified = code
        modified = re.sub(
            rf'Component(?:\s+Type)?\s+{re.escape(target_name)}[^{{]*\{{[^}}]*\}}\s*;?',
            '', modified, flags=re.DOTALL)
        modified = re.sub(rf'\s*Component\s+{re.escape(target_name)}\s*:\s*\w+\s*;', '', modified)
        modified = re.sub(
            rf'\s*{re.escape(target_name)}\s*:\s*(?:process|thread|device|memory|system)\s*\w*\s*;',
            '', modified)
        modified = re.sub(rf'\s*{re.escape(target_name)}\.\w+\s+to\s+\w+\.\w+\s*;', '', modified)
        modified = re.sub(rf'\s*\w+\.\w+\s+to\s+{re.escape(target_name)}\.\w+\s*;', '', modified)
        modified = re.sub(r'Attachments\s*\{\s*\}', '', modified)
        modified = modified.strip()
        if modified == code.strip() or len(modified) < 10:
            continue
        for tmpl in random.sample(PARAPHRASES["DELETE_COMPONENT"], min(3, 7)):
            request = tmpl.format(nom=target_name, system=system)
            s = {"adl_avant" : code,
                 "input"     : fmt_input("DELETE_COMPONENT", code, request),
                 "target"    : modified,
                 "operation" : "DELETE_COMPONENT",
                 "adl_type"  : adl_type,
                 "difficulte": "hard",
                 "cible"     : target_name,
                 "source"    : "comp800_v5"}
            if validate(s):
                results.append(s)
    return results


# ===========================================================================
# Operation 5 — ADD_CONNECTOR
# Applied exclusively to complete system blocks (is_system_block filter).
# Validated by five strict rules (validate_add_connector).
# ===========================================================================
def gen_add_connector(block: dict) -> list:
    """Generate ADD_CONNECTOR samples for complete system-level blocks only."""
    code     = block["adl_code"]
    adl_type = block["adl_type"]
    if not is_system_block(code):
        return []
    system  = extract_system_name(code)
    results = []
    if adl_type == "ACME":
        for conn_type, conn_name in random.sample(CONNECTORS_ACME, min(3, len(CONNECTORS_ACME))):
            decl = f"\n  Connector {conn_name} : {conn_type};"
            if "Attachments" in code:
                modified = re.sub(r'(Attachments\s*\{)', decl + r'\n  \1', code, count=1)
            elif re.search(r'\}\s*$', code):
                modified = re.sub(r'(\})\s*$', decl + r'\n}', code, count=1)
            else:
                modified = code + decl
            if modified == code:
                continue
            for tmpl in random.sample(PARAPHRASES["ADD_CONNECTOR"], min(3, len(PARAPHRASES["ADD_CONNECTOR"]))):
                request = tmpl.format(type=conn_type, nom=conn_name, system=system)
                s = {"adl_avant"  : code,
                     "input"      : fmt_input("ADD_CONNECTOR", code, request),
                     "target"     : modified,
                     "operation"  : "ADD_CONNECTOR",
                     "adl_type"   : adl_type,
                     "difficulte" : "medium",
                     "cible"      : system,
                     "cible_conn" : conn_name,
                     "source"     : "comp800_v5"}
                if validate_add_connector(s):
                    results.append(s)
    else:
        for conn_type, conn_name, _ in random.sample(CONNECTORS_AADL, min(3, len(CONNECTORS_AADL))):
            conn_line = f"    {conn_name} : bus access {conn_type};"
            if re.search(r'\bconnections\b', code, re.IGNORECASE):
                modified = re.sub(
                    r'(\bconnections\b[^;]*;)', r'\1' + f'\n{conn_line}',
                    code, count=1, flags=re.DOTALL)
            elif re.search(r'\bend\s+\w', code, re.IGNORECASE):
                modified = re.sub(
                    r'(\bend\s+\w)', f'  connections\n{conn_line}\n\g<1>',
                    code, count=1, flags=re.IGNORECASE)
            elif re.search(r'\bend\b', code, re.IGNORECASE):
                modified = re.sub(
                    r'(\bend\b)', f'  connections\n{conn_line}\nend',
                    code, count=1, flags=re.IGNORECASE)
            else:
                modified = code.rstrip() + f'\n  connections\n{conn_line}\n'
            if modified == code:
                continue
            for tmpl in random.sample(PARAPHRASES["ADD_CONNECTOR"], min(3, len(PARAPHRASES["ADD_CONNECTOR"]))):
                request = tmpl.format(type=conn_type, nom=conn_name, system=system)
                s = {"adl_avant"  : code,
                     "input"      : fmt_input("ADD_CONNECTOR", code, request),
                     "target"     : modified,
                     "operation"  : "ADD_CONNECTOR",
                     "adl_type"   : adl_type,
                     "difficulte" : "medium",
                     "cible"      : system,
                     "cible_conn" : conn_name,
                     "source"     : "comp800_v5"}
                if validate_add_connector(s):
                    results.append(s)
    return results


# ===========================================================================
# Operation 6 — DELETE_CONNECTOR
# ===========================================================================
def gen_delete_connector(block: dict) -> list:
    """Generate DELETE_CONNECTOR samples by removing an identified connector."""
    code     = block["adl_code"]
    adl_type = block["adl_type"]
    system   = extract_system_name(code)
    if adl_type == "ACME":
        candidates = re.findall(r'Connector(?:\s+Type)?\s+(\w+)', code, re.IGNORECASE)
    else:
        candidates = re.findall(
            r'(\w+)\s*:\s*(?:bus\s+access|event\s+port|data\s+port)\s+\w+', code)
        if not candidates:
            candidates = re.findall(r'(?:bus|event\s+data)\s+(\w+)\b', code)
        if not candidates:
            candidates = re.findall(r'Connector(?:\s+Type)?\s+(\w+)', code, re.IGNORECASE)
    candidates = [c for c in candidates if c.lower() not in STOPWORDS and len(c) > 2]
    if not candidates:
        return []
    results = []
    for target_name in candidates[:3]:
        modified = code
        modified = re.sub(
            rf'Connector(?:\s+Type)?\s+{re.escape(target_name)}[^{{]*\{{[^}}]*\}}\s*;?',
            '', modified, flags=re.DOTALL)
        modified = re.sub(rf'\s*Connector\s+{re.escape(target_name)}\s*:\s*\w+\s*;', '', modified)
        modified = re.sub(
            rf'\s*{re.escape(target_name)}\s*:\s*(?:bus\s+access|event)\s*\w*\s*;', '', modified)
        modified = re.sub(rf'\s*\w+\.\w+\s+to\s+{re.escape(target_name)}\.\w+\s*;', '', modified)
        modified = re.sub(rf'\s*{re.escape(target_name)}\.\w+\s+to\s+\w+\.\w+\s*;', '', modified)
        modified = re.sub(r'Attachments\s*\{\s*\}', '', modified)
        modified = modified.strip()
        if modified == code.strip() or len(modified) < 10:
            continue
        for tmpl in random.sample(PARAPHRASES["DELETE_CONNECTOR"], min(3, 7)):
            request = tmpl.format(nom=target_name, system=system)
            s = {"adl_avant" : code,
                 "input"     : fmt_input("DELETE_CONNECTOR", code, request),
                 "target"    : modified,
                 "operation" : "DELETE_CONNECTOR",
                 "adl_type"  : adl_type,
                 "difficulte": "hard",
                 "cible"     : target_name,
                 "source"    : "comp800_v5"}
            if validate(s):
                results.append(s)
    return results


GENERATORS = [
    ("ADD_PORT",         gen_add_port),
    ("MODIFY_PROPERTY",  gen_modify_property),
    ("ADD_COMPONENT",    gen_add_component),
    ("DELETE_COMPONENT", gen_delete_component),
    ("ADD_CONNECTOR",    gen_add_connector),
    ("DELETE_CONNECTOR", gen_delete_connector),
]
print("All six generators defined.")

## 6. Generation Pipeline

All six generators are applied to every block extracted from the source dataset. Results are deduplicated on `(input, target)` pairs.

In [ ]:
def extract_blocks(df: pd.DataFrame) -> list:
    """Convert DataFrame rows into block dictionaries for the generators."""
    blocks = []
    for _, row in df.iterrows():
        code = str(row["input"]).strip()
        if len(code.split()) >= 4:
            blocks.append({
                "adl_code"     : code,
                "adl_type"     : str(row.get("adl_type", "ACME")),
                "element_type" : str(row.get("element_type", "Component")),
                "quality"      : float(row.get("quality", 3.0)),
            })
    return blocks


blocks = extract_blocks(df)
n_system_blocks = sum(1 for b in blocks if is_system_block(b["adl_code"]))
print(f"Blocks extracted: {len(blocks)}")
print(f"  ACME         : {sum(1 for b in blocks if b['adl_type']=='ACME')}")
print(f"  AADL         : {sum(1 for b in blocks if b['adl_type']=='AADL')}")
print(f"  System blocks (eligible for ADD_CONNECTOR): {n_system_blocks}")

# ── Run all generators ────────────────────────────────────────────────────
all_samples = []
op_counter  = defaultdict(int)

for i, block in enumerate(blocks):
    for op_name, gen in GENERATORS:
        for s in gen(block):
            all_samples.append(s)
            op_counter[op_name] += 1
    if (i + 1) % 100 == 0:
        print(f"  {i+1}/{len(blocks)} blocks processed — {len(all_samples)} samples")

# ── Deduplication ─────────────────────────────────────────────────────────
df_evo = pd.DataFrame(all_samples)
n_raw  = len(df_evo)
df_evo = df_evo.drop_duplicates(subset=["input", "target"])
df_evo = df_evo[df_evo["target"].notna() & (df_evo["target"] != "")]
df_evo = df_evo.reset_index(drop=True)

print(f"\nGeneration complete:")
print(f"  Raw samples    : {n_raw}")
print(f"  After dedup    : {len(df_evo)}")
print(f"  Duplicates     : {n_raw - len(df_evo)}")
print(f"\nDistribution by operation:")
for op, cnt in df_evo["operation"].value_counts().items():
    print(f"  {op:<25} : {cnt:>5} ({cnt/len(df_evo)*100:.1f}%)")

## 7. Synthetic Boost for DELETE Operations

DELETE_COMPONENT and DELETE_CONNECTOR are the least frequently generated operations because deletions require the target entity to already exist in the source snippet. To ensure adequate training coverage, 200 ACME and 200 AADL system blocks are generated synthetically and processed exclusively through the two DELETE generators.

Synthetic samples are tagged `source = 'synthetic_v5'` and capped at 200 per operation during the balancing step to limit memorisation risk.

In [ ]:
# ===========================================================================
# Synthetic system block templates
# ===========================================================================
TEMPLATES_ACME = [
    """System {sys} = {{
  Component {c1} : {t1};
  Component {c2} : {t2};
  Connector {conn} : {tc};
  Attachments {{
    {c1}.send to {conn}.caller;
    {c2}.receive to {conn}.callee;
  }}
}}""",
    """System {sys} = {{
  Component {c1} : {t1};
  Component {c2} : {t2};
  Component {c3} : {t3};
  Connector {conn} : {tc};
  Attachments {{
    {c1}.out to {conn}.source;
    {c2}.in  to {conn}.sink;
    {c3}.log to {conn}.monitor;
  }}
}}""",
    """System {sys} = {{
  Component {c1} : {t1};
  Component {c2} : {t2};
  Connector {conn1} : {tc};
  Connector {conn2} : {tc2};
  Attachments {{
    {c1}.request to {conn1}.caller;
    {c2}.response to {conn2}.callee;
  }}
}}""",
]

TEMPLATES_AADL = [
    """system implementation {sys}.impl
  subcomponents
    {c1}: process {t1};
    {c2}: process {t2};
  connections
    {conn}: bus access {tc};
end {sys}.impl;""",
    """system implementation {sys}.impl
  subcomponents
    {c1}: thread {t1};
    {c2}: thread {t2};
    {c3}: process {t3};
  connections
    {conn}: bus access {tc};
  properties
    Security_Level => medium;
end {sys}.impl;""",
    """system implementation {sys}.impl
  subcomponents
    {c1}: process {t1};
    {c2}: process {t2};
    {c3}: thread {t3};
  connections
    {conn}: bus access {tc};
    {conn2}: bus access {tc2};
end {sys}.impl;""",
]

SYSTEM_NAMES = [
    "AppSystem", "CtrlSystem", "DataSystem", "MonitorSystem",
    "SafetySystem", "NetSystem", "EmbeddedSystem", "RealTimeSystem",
    "FlightSystem", "NavSystem", "SensorSystem", "ActuatorSystem",
    "ProcessSystem", "SchedulerSystem", "LogicSystem", "CoreSystem",
]
COMP_NAMES = [
    "processor", "controller", "monitor", "logger", "sensor",
    "actuator", "gateway", "cacheUnit", "validator", "dispatcher",
    "scheduler", "handler", "router", "aggregator", "transformer",
    "publisher", "subscriber", "coordinator", "balancer", "analyzer",
]
COMP_TYPES = [n[0].upper() + n[1:] for n in COMP_NAMES]
CONN_NAMES = [
    "dataLink", "ctrlBus", "eventChan", "msgPipe",
    "rpcConn", "asyncLink", "syncBus", "streamPipe",
    "monitorLink", "auditBus", "healthChan", "serviceBus",
]
CONN_TYPES = [n[0].upper() + n[1:] for n in CONN_NAMES]


def make_synthetic_acme(seed: int) -> dict:
    """Generate a syntactically valid ACME system block."""
    r    = random.Random(seed)
    tmpl = r.choice(TEMPLATES_ACME)
    noms = r.sample(COMP_NAMES, 3)
    typs = r.sample(COMP_TYPES, 3)
    sys  = r.choice(SYSTEM_NAMES)
    i1, i2 = r.randint(0, len(CONN_NAMES)-1), r.randint(0, len(CONN_NAMES)-1)
    code = tmpl.format(
        sys=sys, c1=noms[0], t1=typs[0], c2=noms[1], t2=typs[1],
        c3=noms[2], t3=typs[2],
        conn=CONN_NAMES[i1], tc=CONN_TYPES[i1],
        conn1=CONN_NAMES[i1], conn2=CONN_NAMES[i2], tc2=CONN_TYPES[i2],
    )
    return {"adl_code": code, "adl_type": "ACME", "element_type": "System", "quality": 4.5}


def make_synthetic_aadl(seed: int) -> dict:
    """Generate a syntactically valid AADL system block."""
    r    = random.Random(seed + 5000)
    tmpl = r.choice(TEMPLATES_AADL)
    noms = r.sample(COMP_NAMES, 3)
    typs = r.sample(COMP_TYPES, 3)
    sys  = r.choice(SYSTEM_NAMES)
    i1, i2 = r.randint(0, len(CONN_NAMES)-1), r.randint(0, len(CONN_NAMES)-1)
    code = tmpl.format(
        sys=sys, c1=noms[0], t1=typs[0], c2=noms[1], t2=typs[1],
        c3=noms[2], t3=typs[2],
        conn=CONN_NAMES[i1], tc=CONN_TYPES[i1],
        conn2=CONN_NAMES[i2], tc2=CONN_TYPES[i2],
    )
    return {"adl_code": code, "adl_type": "AADL", "element_type": "system", "quality": 4.5}


NB_SYNTHETIC = 200
synthetic_blocks = (
    [make_synthetic_acme(i) for i in range(NB_SYNTHETIC)] +
    [make_synthetic_aadl(i) for i in range(NB_SYNTHETIC)]
)
print(f"Synthetic blocks generated: {len(synthetic_blocks)}")

# Apply DELETE generators only
delete_samples = []
for block in synthetic_blocks:
    for s in gen_delete_component(block):
        s["source"] = "synthetic_v5"
        delete_samples.append(s)
    for s in gen_delete_connector(block):
        s["source"] = "synthetic_v5"
        delete_samples.append(s)

df_synthetic = pd.DataFrame(delete_samples).drop_duplicates(
    subset=["input", "target"]).reset_index(drop=True)
print(f"Synthetic DELETE samples after dedup:")
for op, cnt in df_synthetic["operation"].value_counts().items():
    print(f"  {op:<25} : {cnt}")

# Merge with the main dataset
df_evo = pd.concat([df_evo, df_synthetic]).drop_duplicates(
    subset=["input", "target"]).reset_index(drop=True)
n_raw  = len(df_evo)
print(f"\nDataset after synthetic boost: {len(df_evo)} total samples")
print("Distribution by operation:")
for op, cnt in df_evo["operation"].value_counts().items():
    print(f"  {op:<25} : {cnt}")

## 8. Class Balancing

Samples are capped per operation to prevent dominant classes from biasing training. Real samples are capped at 400 per operation; synthetic samples are capped at 200 per operation.

In [ ]:
QUOTA_REAL      = 400
QUOTA_SYNTHETIC = 200

df_real  = df_evo[df_evo["source"] != "synthetic_v5"].copy()
df_synth = df_evo[df_evo["source"] == "synthetic_v5"].copy()

groups = []
for op in df_evo["operation"].unique():
    g_real = df_real[df_real["operation"] == op].copy()
    if len(g_real) > QUOTA_REAL:
        g_real = g_real.sample(n=QUOTA_REAL, random_state=42)
    g_synth = df_synth[df_synth["operation"] == op].copy()
    if len(g_synth) > 0:
        n_synth = min(len(g_synth), QUOTA_SYNTHETIC)
        g_synth = g_synth.sample(n=n_synth, random_state=42)
        groups.append(pd.concat([g_real, g_synth]))
    else:
        groups.append(g_real)

df_balanced = pd.concat(groups).sample(frac=1, random_state=42).reset_index(drop=True)
nb_final    = len(df_balanced)
taux_val    = nb_final / n_raw * 100

print(f"Balanced dataset: {nb_final} samples")
print(f"Distribution by operation:")
for op, cnt in df_balanced["operation"].value_counts().items():
    n_real  = len(df_balanced[(df_balanced["operation"]==op) & (df_balanced["source"]!="synthetic_v5")])
    n_synth = cnt - n_real
    synth_note = f" (incl. {n_synth} synthetic)" if n_synth > 0 else ""
    print(f"  {op:<25} : {cnt:>5}{synth_note}")
print(f"\nPipeline validity rate: {taux_val:.1f}%")

## 9. Stratified Split and Export

The balanced dataset is split into train / validation / test subsets (80 / 10 / 10) with stratification by operation. A minimum of 20 samples per operation is guaranteed in the test set.

In [ ]:
TEST_MIN_PER_OP = 20

test_parts      = []
train_val_parts = []

for op in df_balanced["operation"].unique():
    group   = df_balanced[df_balanced["operation"] == op].copy()
    n_total = len(group)
    n_test  = max(TEST_MIN_PER_OP, int(n_total * 0.10))
    n_test  = min(n_test, int(n_total * 0.40), n_total - 10)
    n_test  = max(n_test, 0)
    if n_test > 0:
        test_g     = group.sample(n=n_test, random_state=42)
        train_val_g = group.drop(test_g.index)
    else:
        test_g      = group.iloc[:0]
        train_val_g = group
    test_parts.append(test_g)
    train_val_parts.append(train_val_g)

df_test      = pd.concat(test_parts).sample(frac=1, random_state=42).reset_index(drop=True)
df_train_val = pd.concat(train_val_parts).sample(frac=1, random_state=42).reset_index(drop=True)

df_train, df_val = train_test_split(
    df_train_val, test_size=0.111,
    stratify=df_train_val["operation"], random_state=42
)

print("Split summary:")
for label, split in [("TRAIN", df_train), ("VAL", df_val), ("TEST", df_test)]:
    print(f"  {label:<6}: {len(split):>5} samples")
    for op, cnt in split["operation"].value_counts().items():
        flag = "OK" if (label != "TEST" or cnt >= TEST_MIN_PER_OP) else "LOW"
        print(f"    [{flag}] {op:<25}: {cnt}")


# ===========================================================================
# Export all splits and the structured dataset report
# ===========================================================================
OUTPUT_COLUMNS = ["input", "target", "operation", "adl_type", "difficulte", "source"]


def save_split(df_split: pd.DataFrame, name: str, out_dir: str):
    """Save a split as both CSV and JSONL."""
    cols = [c for c in OUTPUT_COLUMNS if c in df_split.columns]
    df_split[cols].to_csv(f"{out_dir}/{name}.csv", index=False, encoding="utf-8")
    with open(f"{out_dir}/{name}.jsonl", "w", encoding="utf-8") as f:
        for _, row in df_split.iterrows():
            f.write(json.dumps({
                "input"      : str(row["input"]),
                "target"     : str(row["target"]),
                "operation"  : str(row["operation"]),
                "adl_type"   : str(row["adl_type"]),
                "difficulte" : str(row["difficulte"]),
                "source"     : str(row.get("source", "")),
            }, ensure_ascii=False) + "\n")
    print(f"  Saved {name:<20}: {len(df_split)} samples")


df_balanced.to_csv(f"{EVO_DIR}/dataset_evolution.csv", index=False, encoding="utf-8")
save_split(df_train, "evo_train", EVO_DIR)
save_split(df_val,   "evo_val",   EVO_DIR)
save_split(df_test,  "evo_test",  EVO_DIR)

# Structured dataset report (suitable for public release)
report = {
    "version"                 : "v5",
    "source_dataset"          : "dataset_Comprehension.csv",
    "seed"                    : 42,
    "total_raw_generated"     : int(n_raw),
    "total_after_balancing"   : int(nb_final),
    "pipeline_validity_rate"  : round(taux_val, 2),
    "splits": {
        "train" : int(len(df_train)),
        "val"   : int(len(df_val)),
        "test"  : int(len(df_test)),
    },
    "split_strategy"          : "stratified by operation, min 20 samples/op in test",
    "test_min_per_operation"  : TEST_MIN_PER_OP,
    "by_operation"            : df_balanced["operation"].value_counts().to_dict(),
    "by_adl_type"             : df_balanced["adl_type"].value_counts().to_dict(),
    "by_difficulty"           : df_balanced["difficulte"].value_counts().to_dict(),
    "by_source": {
        "real"      : int(len(df_balanced[df_balanced["source"] != "synthetic_v5"])),
        "synthetic" : int(len(df_balanced[df_balanced["source"] == "synthetic_v5"])),
    },
    "taxonomy": {
        "interface_level"  : ["ADD_PORT", "MODIFY_PROPERTY"],
        "component_level"  : ["ADD_COMPONENT", "DELETE_COMPONENT"],
        "connector_level"  : ["ADD_CONNECTOR", "DELETE_CONNECTOR"]
    },
    "adl_formalisms"          : ["ACME", "AADL"],
    "input_format"            : "evolve: [OP] {operation} [ADL] {adl_code} [REQ] {request}",
    "validation_rules": {
        "generic"        : ["non-trivial target", "target != source",
                             "balanced braces (ACME)", "deleted entity absent from target",
                             "standard [OP][ADL][REQ] format"],
        "add_connector"  : ["target >= 10 tokens", "target != source",
                             "connector name present in target",
                             "connector keyword present", "len(target) > len(source)"]
    }
}
with open(f"{EVO_DIR}/rapport_evolution_v5.json", "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)
print(f"\nrapport_evolution_v5.json saved.")

## 10. Integrity Verification

All three JSONL files are validated against five structural constraints. Any violation is reported with the line number and error message.

In [ ]:
print("Files in output directory:")
for fname in sorted(os.listdir(EVO_DIR)):
    path = f"{EVO_DIR}/{fname}"
    size = os.path.getsize(path) / 1024
    if fname.endswith(".jsonl"):
        with open(path) as fp:
            nb = sum(1 for _ in fp)
        print(f"  {fname:<45}: {nb:>5} lines  ({size:.1f} KB)")
    else:
        print(f"  {fname:<45}: ({size:.1f} KB)")

print("\nJSONL integrity validation:")
total_errors = 0
for split_name in ["evo_train", "evo_val", "evo_test"]:
    errors = 0
    with open(f"{EVO_DIR}/{split_name}.jsonl") as f:
        for i, line in enumerate(f):
            try:
                obj = json.loads(line)
                assert all(k in obj for k in
                           ["input", "target", "operation", "adl_type", "difficulte"])
                assert obj["input"] != obj["target"]
                assert len(obj["target"].strip()) > 5
                assert all(t in obj["input"] for t in ["[OP]", "[ADL]", "[REQ]"])
                if obj["operation"] == "ADD_CONNECTOR":
                    tgt = obj["target"].lower()
                    assert any(kw in tgt for kw in
                               ["connector", "connections", "bus access"]), \
                        "ADD_CONNECTOR target missing connector keyword"
            except Exception as e:
                print(f"  ERROR {split_name} line {i}: {e}")
                errors += 1
    status = "PASS" if errors == 0 else f"FAIL ({errors} errors)"
    print(f"  {split_name:<20}: {status}")
    total_errors += errors

print(f"\nFinal report:")
print(f"  Total samples : {nb_final}")
print(f"  Train / Val / Test : {len(df_train)} / {len(df_val)} / {len(df_test)}")
print(f"  JSONL errors  : {total_errors}")
print(f"  Output        : {EVO_DIR}")